In [1]:
!pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 12.4 MB/s eta 0:00:00


In [2]:
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from getpass import getpass

uri = getpass("Paste MongoDB connection string here: ")

client = MongoClient(uri, server_api=ServerApi("1"))
client.admin.command("ping")

print("MongoDB Atlas connected successfully")

Paste MongoDB connection string here: ··········
MongoDB Atlas connected successfully


In [3]:
db = client["northstar_db"]

print("Database selected: northstar_db")

Database selected: northstar_db


In [4]:
from google.colab import files

uploaded = files.upload()

Saving northstar_dataset.zip to northstar_dataset.zip


In [5]:
import zipfile
import os
import glob

zip_files = glob.glob("/content/*.zip")
print("ZIP files found:", zip_files)

zip_path = zip_files[0]
extract_path = "/content/northstar_data"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully")

for root, dirs, files in os.walk(extract_path):
    print(root)
    print(files)

ZIP files found: ['/content/northstar_dataset.zip']
Dataset extracted successfully
/content/northstar_data
[]
/content/northstar_data/northstar_dataset
['deliveries.csv', 'orders.csv', 'hubs.csv', 'customers.csv', 'app_events.csv', 'data_dictionary.csv', 'complaints.csv', 'incidents.csv', 'vehicles.csv', 'README.txt', 'drivers.csv']


In [6]:
import pandas as pd
import json
import glob
from pprint import pprint

def find_csv(filename):
    matches = glob.glob(f"/content/northstar_data/**/{filename}", recursive=True)
    if len(matches) == 0:
        raise FileNotFoundError(f"{filename} not found")
    return matches[0]

orders = pd.read_csv(find_csv("orders.csv"))
deliveries = pd.read_csv(find_csv("deliveries.csv"))
customers = pd.read_csv(find_csv("customers.csv"))
complaints = pd.read_csv(find_csv("complaints.csv"))
incidents = pd.read_csv(find_csv("incidents.csv"))
drivers = pd.read_csv(find_csv("drivers.csv"))
vehicles = pd.read_csv(find_csv("vehicles.csv"))
hubs = pd.read_csv(find_csv("hubs.csv"))
app_events = pd.read_csv(find_csv("app_events.csv"))

print("All NorthStar CSV files loaded successfully")
print("Orders:", orders.shape)
print("Deliveries:", deliveries.shape)
print("Customers:", customers.shape)
print("Complaints:", complaints.shape)
print("Incidents:", incidents.shape)
print("Drivers:", drivers.shape)
print("Vehicles:", vehicles.shape)
print("Hubs:", hubs.shape)
print("App Events:", app_events.shape)

All NorthStar CSV files loaded successfully
Orders: (1250, 11)
Deliveries: (950, 13)
Customers: (650, 9)
Complaints: (320, 10)
Incidents: (280, 7)
Drivers: (170, 8)
Vehicles: (120, 8)
Hubs: (8, 5)
App Events: (640, 10)


In [7]:
def df_to_docs(df):
    return json.loads(df.to_json(orient="records"))

orders_docs = df_to_docs(orders)
deliveries_docs = df_to_docs(deliveries)
customers_docs = df_to_docs(customers)
complaints_docs = df_to_docs(complaints)
incidents_docs = df_to_docs(incidents)
drivers_docs = df_to_docs(drivers)
vehicles_docs = df_to_docs(vehicles)
hubs_docs = df_to_docs(hubs)
app_events_docs = df_to_docs(app_events)

print("Data converted into MongoDB document format")
print("Sample order document:")
pprint(orders_docs[0])

Data converted into MongoDB document format
Sample order document:
{'booking_channel': 'App',
 'customer_id': 'C0292',
 'dropoff_zone': 'South',
 'order_created_at': '2024-08-20 14:43:00',
 'order_id': 'O00001',
 'order_value': 126.65,
 'pickup_zone': 'Airport',
 'priority_level': 'Medium',
 'promised_window_hours': 6,
 'service_type': 'Passenger',
 'special_handling_flag': 0}


In [8]:
collection_names = [
    "orders",
    "deliveries",
    "customers",
    "complaints",
    "incidents",
    "drivers",
    "vehicles",
    "hubs",
    "app_events"
]

# Clear old data first
for name in collection_names:
    db[name].delete_many({})

# Insert fresh data
db.orders.insert_many(orders_docs)
db.deliveries.insert_many(deliveries_docs)
db.customers.insert_many(customers_docs)
db.complaints.insert_many(complaints_docs)
db.incidents.insert_many(incidents_docs)
db.drivers.insert_many(drivers_docs)
db.vehicles.insert_many(vehicles_docs)
db.hubs.insert_many(hubs_docs)
db.app_events.insert_many(app_events_docs)

print("Data inserted successfully into MongoDB")

for name in collection_names:
    print(name, ":", db[name].count_documents({}))

Data inserted successfully into MongoDB
orders : 1250
deliveries : 950
customers : 650
complaints : 320
incidents : 280
drivers : 170
vehicles : 120
hubs : 8
app_events : 640


In [9]:
def make_dict(df, key):
    docs = df_to_docs(df)
    return {doc[key]: doc for doc in docs if key in doc and doc[key] is not None}

def group_records(df, key):
    output = {}
    for value, group in df.groupby(key, dropna=True):
        output[value] = df_to_docs(group.drop(columns=[key], errors="ignore"))
    return output

customers_dict = make_dict(customers, "customer_id")
drivers_dict = make_dict(drivers, "driver_id")
vehicles_dict = make_dict(vehicles, "vehicle_id")
hubs_dict = make_dict(hubs, "hub_id")

deliveries_by_order = group_records(deliveries, "order_id")
complaints_by_order = group_records(complaints, "order_id")
incidents_by_delivery = group_records(incidents, "delivery_id")

app_events_with_order = app_events.dropna(subset=["order_id"])
app_events_by_order = group_records(app_events_with_order, "order_id")

print("Related records grouped successfully")

Related records grouped successfully


In [10]:
orders_list = df_to_docs(orders)

integrated_docs = []

for order in orders_list:
    order_id = order.get("order_id")
    customer_id = order.get("customer_id")

    delivery_list = deliveries_by_order.get(order_id, [])
    delivery = delivery_list[0] if len(delivery_list) > 0 else None

    delivery_id = delivery.get("delivery_id") if delivery else None
    driver_id = delivery.get("driver_id") if delivery else None
    vehicle_id = delivery.get("vehicle_id") if delivery else None
    hub_id = delivery.get("hub_id") if delivery else None

    customer = customers_dict.get(customer_id)
    driver = drivers_dict.get(driver_id)
    vehicle = vehicles_dict.get(vehicle_id)
    hub = hubs_dict.get(hub_id)

    complaint_list = complaints_by_order.get(order_id, [])
    app_event_list = app_events_by_order.get(order_id, [])
    incident_list = incidents_by_delivery.get(delivery_id, []) if delivery_id else []

    is_failed = False
    if delivery:
        is_failed = delivery.get("delivery_status") == "Failed"

    has_complaint = len(complaint_list) > 0
    has_incident = len(incident_list) > 0
    high_risk = is_failed or has_complaint or has_incident

    cost_per_km = None
    if delivery:
        distance = delivery.get("route_distance_km")
        cost = delivery.get("fuel_or_charge_cost")
        if distance and distance != 0:
            cost_per_km = round(cost / distance, 2)

    case_doc = {
        "order_id": order_id,
        "customer_id": customer_id,
        "service_type": order.get("service_type"),
        "priority_level": order.get("priority_level"),
        "order_value": order.get("order_value"),
        "booking_channel": order.get("booking_channel"),

        "zones": {
            "pickup_zone": order.get("pickup_zone"),
            "dropoff_zone": order.get("dropoff_zone"),
            "customer_home_zone": customer.get("home_zone") if customer else None
        },

        "customer": customer,
        "delivery": delivery,
        "hub": hub,
        "driver": driver,
        "vehicle": vehicle,
        "complaints": complaint_list,
        "incidents": incident_list,
        "app_events": app_event_list,

        "analytics_flags": {
            "is_failed": is_failed,
            "has_complaint": has_complaint,
            "has_incident": has_incident,
            "high_risk": high_risk,
            "cost_per_km": cost_per_km,
            "complaint_count": len(complaint_list),
            "incident_count": len(incident_list),
            "app_event_count": len(app_event_list)
        }
    }

    integrated_docs.append(case_doc)

print("Integrated service case documents created:", len(integrated_docs))
print("Sample integrated document:")
pprint(integrated_docs[0])

Integrated service case documents created: 1250
Sample integrated document:
{'analytics_flags': {'app_event_count': 1,
                     'complaint_count': 0,
                     'cost_per_km': 0.59,
                     'has_complaint': False,
                     'has_incident': False,
                     'high_risk': False,
                     'incident_count': 0,
                     'is_failed': False},
 'app_events': [{'api_latency_ms': 204,
                 'customer_id': 'C0112',
                 'device_type': 'Android',
                 'event_id': 'AE00503',
                 'event_timestamp': '2024-08-02 12:35:00',
                 'event_type': 'delivery_instruction_update',
                 'session_id': 'S44209',
                 'success_flag': 1,
                 'zone_context': 'Riverside'}],
 'booking_channel': 'App',
 'complaints': [],
 'customer': {'account_status': 'Active',
              'age': 24,
              'app_engagement_score': 57.9,
              '

In [11]:
db.integrated_service_cases.delete_many({})

db.integrated_service_cases.insert_many(integrated_docs)

print("Integrated service cases inserted:", db.integrated_service_cases.count_documents({}))

Integrated service cases inserted: 1250


In [12]:
sample_case = db.integrated_service_cases.find_one({}, {"_id": 0})

pprint(sample_case)

{'analytics_flags': {'app_event_count': 1,
                     'complaint_count': 0,
                     'cost_per_km': 0.59,
                     'has_complaint': False,
                     'has_incident': False,
                     'high_risk': False,
                     'incident_count': 0,
                     'is_failed': False},
 'app_events': [{'api_latency_ms': 204,
                 'customer_id': 'C0112',
                 'device_type': 'Android',
                 'event_id': 'AE00503',
                 'event_timestamp': '2024-08-02 12:35:00',
                 'event_type': 'delivery_instruction_update',
                 'session_id': 'S44209',
                 'success_flag': 1,
                 'zone_context': 'Riverside'}],
 'booking_channel': 'App',
 'complaints': [],
 'customer': {'account_status': 'Active',
              'age': 24,
              'app_engagement_score': 57.9,
              'customer_id': 'C0292',
              'customer_type': 'Consumer',
          

In [13]:
test_case = {
    "order_id": "TEST_ORDER_001",
    "customer_id": "TEST_CUSTOMER_001",
    "service_type": "LastMileDelivery",
    "priority_level": "High",
    "order_value": 150.00,
    "booking_channel": "App",

    "delivery": {
        "delivery_status": "Failed",
        "route_distance_km": 12.5,
        "manual_route_override_count": 3,
        "customer_rating_post_delivery": 2.1,
        "fuel_or_charge_cost": 18.75
    },

    "complaints": [
        {
            "complaint_type": "LateArrival",
            "severity": "High",
            "status": "Open",
            "compensation_amount": 25.00
        }
    ],

    "analytics_flags": {
        "is_failed": True,
        "has_complaint": True,
        "has_incident": False,
        "high_risk": True
    }
}

db.integrated_service_cases.insert_one(test_case)

print("CREATE completed: test service case inserted")

CREATE completed: test service case inserted


In [14]:
read_result = db.integrated_service_cases.find_one(
    {"order_id": "TEST_ORDER_001"},
    {"_id": 0}
)

pprint(read_result)

{'analytics_flags': {'has_complaint': True,
                     'has_incident': False,
                     'high_risk': True,
                     'is_failed': True},
 'booking_channel': 'App',
 'complaints': [{'compensation_amount': 25.0,
                 'complaint_type': 'LateArrival',
                 'severity': 'High',
                 'status': 'Open'}],
 'customer_id': 'TEST_CUSTOMER_001',
 'delivery': {'customer_rating_post_delivery': 2.1,
              'delivery_status': 'Failed',
              'fuel_or_charge_cost': 18.75,
              'manual_route_override_count': 3,
              'route_distance_km': 12.5},
 'order_id': 'TEST_ORDER_001',
 'order_value': 150.0,
 'priority_level': 'High',
 'service_type': 'LastMileDelivery'}


In [15]:
db.integrated_service_cases.update_one(
    {"order_id": "TEST_ORDER_001"},
    {
        "$set": {
            "case_review_status": "Manager Review Required",
            "analytics_flags.high_risk": True
        }
    }
)

updated_result = db.integrated_service_cases.find_one(
    {"order_id": "TEST_ORDER_001"},
    {"_id": 0}
)

pprint(updated_result)

{'analytics_flags': {'has_complaint': True,
                     'has_incident': False,
                     'high_risk': True,
                     'is_failed': True},
 'booking_channel': 'App',
 'case_review_status': 'Manager Review Required',
 'complaints': [{'compensation_amount': 25.0,
                 'complaint_type': 'LateArrival',
                 'severity': 'High',
                 'status': 'Open'}],
 'customer_id': 'TEST_CUSTOMER_001',
 'delivery': {'customer_rating_post_delivery': 2.1,
              'delivery_status': 'Failed',
              'fuel_or_charge_cost': 18.75,
              'manual_route_override_count': 3,
              'route_distance_km': 12.5},
 'order_id': 'TEST_ORDER_001',
 'order_value': 150.0,
 'priority_level': 'High',
 'service_type': 'LastMileDelivery'}


In [16]:
db.integrated_service_cases.delete_one(
    {"order_id": "TEST_ORDER_001"}
)

deleted_check = db.integrated_service_cases.find_one(
    {"order_id": "TEST_ORDER_001"}
)

print("Deleted record check:", deleted_check)

Deleted record check: None


In [17]:
pipeline_hub_performance = [
    {
        "$group": {
            "_id": "$hub.hub_name",
            "total_orders": {"$sum": 1},
            "failed_orders": {
                "$sum": {
                    "$cond": ["$analytics_flags.is_failed", 1, 0]
                }
            },
            "complaint_cases": {
                "$sum": {
                    "$cond": ["$analytics_flags.has_complaint", 1, 0]
                }
            },
            "incident_cases": {
                "$sum": {
                    "$cond": ["$analytics_flags.has_incident", 1, 0]
                }
            },
            "average_cost": {"$avg": "$delivery.fuel_or_charge_cost"},
            "average_rating": {"$avg": "$delivery.customer_rating_post_delivery"}
        }
    },
    {
        "$addFields": {
            "failed_rate_percent": {
                "$round": [
                    {
                        "$multiply": [
                            {"$divide": ["$failed_orders", "$total_orders"]},
                            100
                        ]
                    },
                    2
                ]
            }
        }
    },
    {
        "$sort": {
            "failed_rate_percent": -1
        }
    }
]

hub_results = list(db.integrated_service_cases.aggregate(pipeline_hub_performance))

for result in hub_results:
    pprint(result)

{'_id': 'Midtown Relay',
 'average_cost': 11.708203125,
 'average_rating': 3.88456,
 'complaint_cases': 31,
 'failed_orders': 26,
 'failed_rate_percent': 20.31,
 'incident_cases': 32,
 'total_orders': 128}
{'_id': 'Central Core',
 'average_cost': 13.686000000000002,
 'average_rating': 3.669557522123894,
 'complaint_cases': 28,
 'failed_orders': 23,
 'failed_rate_percent': 20.0,
 'incident_cases': 34,
 'total_orders': 115}
{'_id': 'Airport Hub',
 'average_cost': 13.319230769230769,
 'average_rating': 3.882135922330097,
 'complaint_cases': 21,
 'failed_orders': 15,
 'failed_rate_percent': 14.42,
 'incident_cases': 25,
 'total_orders': 104}
{'_id': 'West Gate',
 'average_cost': 13.167007874015749,
 'average_rating': 3.9154761904761908,
 'complaint_cases': 26,
 'failed_orders': 16,
 'failed_rate_percent': 12.6,
 'incident_cases': 33,
 'total_orders': 127}
{'_id': 'North Exchange',
 'average_cost': 12.755808823529412,
 'average_rating': 3.840592592592593,
 'complaint_cases': 26,
 'failed_or

In [18]:
db.integrated_service_cases.drop_indexes()

print("Existing custom indexes dropped")

query = {
    "analytics_flags.high_risk": True
}

explain_before = db.command(
    "explain",
    {
        "find": "integrated_service_cases",
        "filter": query
    },
    verbosity="executionStats"
)

print("BEFORE INDEXING")
print("Documents examined:", explain_before["executionStats"]["totalDocsExamined"])
print("Keys examined:", explain_before["executionStats"]["totalKeysExamined"])
print("Execution time ms:", explain_before["executionStats"]["executionTimeMillis"])
print("Winning plan:")
pprint(explain_before["queryPlanner"]["winningPlan"])

Existing custom indexes dropped
BEFORE INDEXING
Documents examined: 1250
Keys examined: 0
Execution time ms: 1
Winning plan:
{'direction': 'forward',
 'filter': {'analytics_flags.high_risk': {'$eq': True}},
 'isCached': False,
 'stage': 'COLLSCAN'}


In [19]:
db.integrated_service_cases.create_index([("analytics_flags.high_risk", 1)])
db.integrated_service_cases.create_index([("order_id", 1)])
db.integrated_service_cases.create_index([("customer_id", 1)])
db.integrated_service_cases.create_index([("service_type", 1)])
db.integrated_service_cases.create_index([("delivery.delivery_status", 1)])
db.integrated_service_cases.create_index([("hub.hub_name", 1)])

print("Indexes created successfully")

for index in db.integrated_service_cases.list_indexes():
    pprint(index)

Indexes created successfully
SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')])
SON([('v', 2), ('key', SON([('analytics_flags.high_risk', 1)])), ('name', 'analytics_flags.high_risk_1')])
SON([('v', 2), ('key', SON([('order_id', 1)])), ('name', 'order_id_1')])
SON([('v', 2), ('key', SON([('customer_id', 1)])), ('name', 'customer_id_1')])
SON([('v', 2), ('key', SON([('service_type', 1)])), ('name', 'service_type_1')])
SON([('v', 2), ('key', SON([('delivery.delivery_status', 1)])), ('name', 'delivery.delivery_status_1')])
SON([('v', 2), ('key', SON([('hub.hub_name', 1)])), ('name', 'hub.hub_name_1')])


In [20]:
explain_after = db.command(
    "explain",
    {
        "find": "integrated_service_cases",
        "filter": query
    },
    verbosity="executionStats"
)

print("AFTER INDEXING")
print("Documents examined:", explain_after["executionStats"]["totalDocsExamined"])
print("Keys examined:", explain_after["executionStats"]["totalKeysExamined"])
print("Execution time ms:", explain_after["executionStats"]["executionTimeMillis"])
print("Winning plan:")
pprint(explain_after["queryPlanner"]["winningPlan"])

AFTER INDEXING
Documents examined: 562
Keys examined: 562
Execution time ms: 2
Winning plan:
{'inputStage': {'direction': 'forward',
                'indexBounds': {'analytics_flags.high_risk': ['[true, true]']},
                'indexName': 'analytics_flags.high_risk_1',
                'indexVersion': 2,
                'isMultiKey': False,
                'isPartial': False,
                'isSparse': False,
                'isUnique': False,
                'keyPattern': {'analytics_flags.high_risk': 1},
                'multiKeyPaths': {'analytics_flags.high_risk': []},
                'stage': 'IXSCAN'},
 'isCached': False,
 'stage': 'FETCH'}


In [21]:
comparison = pd.DataFrame({
    "Metric": [
        "Documents Examined",
        "Keys Examined",
        "Execution Time ms"
    ],
    "Before Indexing": [
        explain_before["executionStats"]["totalDocsExamined"],
        explain_before["executionStats"]["totalKeysExamined"],
        explain_before["executionStats"]["executionTimeMillis"]
    ],
    "After Indexing": [
        explain_after["executionStats"]["totalDocsExamined"],
        explain_after["executionStats"]["totalKeysExamined"],
        explain_after["executionStats"]["executionTimeMillis"]
    ]
})

comparison

,Metric,Before Indexing,After Indexing
0,Documents Examined,1250,562
1,Keys Examined,0,562
2,Execution Time ms,1,2


In [22]:
print("Final MongoDB collections:")

for name in db.list_collection_names():
    print(name, ":", db[name].count_documents({}))

Final MongoDB collections:
orders : 1250
deliveries : 950
complaints : 320
hubs : 8
vehicles : 120
app_events : 640
incidents : 280
drivers : 170
customers : 650
integrated_service_cases : 1250
